# 统一参数与论文绘图

主目录：`results/figures/publication`。正式排版使用SVG；PDF可用于LaTeX；PNG仅供Notebook预览。作者框架SVG/PPTX保持不变。

ESGR为表达—空间引导选路（Expression–Spatial Guided Routing）的正式名称；历史规则文件和实验结果不改名。

本轮升级为矢量数据点、路径和可编辑文字。组织学背景是原始照片，仍嵌入位图；不会插值补点、平滑类别或改变ARI。

正文展示14项方法：SiDMGF只使用保留通路图与融合结构的完整模型，显示名称统一为SiDMGF。空间图对照及原论文指标保存在 `results/figures/supplementary_controls`，不进入正文图表。方法配色统一，区域颜色在Ground Truth和各预测之间保持一致。UMAP每行一种方法，左右为参考与预测着色，共用投影坐标；每页最多5种方法，末页按实际方法数排版，无空白格。

直接运行全部单元格。代码或配色改变会重新出图；已有UMAP缓存仍可复用，不重新训练。本轮Word只预留SVG图位，待出图完成后再插入。


In [ ]:
from pathlib import Path
import sys, importlib
ROOT = next((q for p in [Path.cwd(), *Path.cwd().parents] for q in [p,p/"sidmgf_final"] if (q/"config/publication_figures.json").is_file()),None)
if ROOT is None: raise FileNotFoundError("未找到 sidmgf_final/config/publication_figures.json")
ROOT = ROOT.resolve()
sys.path.insert(0,str(ROOT))
import src
if Path(src.__file__).resolve().parent != ROOT/"src":
    raise RuntimeError("当前kernel已加载另一个项目的src模块。请Restart Kernel，然后从本Notebook首格开始运行。")
from src import plot_style, figures, publication_style, publication_figures as plots
importlib.reload(publication_style)
importlib.reload(plot_style)
importlib.reload(figures)
importlib.reload(plots)
print("解释器:", sys.executable)
print("当前绘图脚本:", Path(plots.__file__).resolve())
print("当前图片目录:", plots.OUT)
print("本入口不会训练模型，也不会自动替换Word内嵌图片。")


In [ ]:
FORCE_REDRAW = False  # 核验后复用；若代码或数据改变，会自动重画。
seed_results, images = plots.generate(force=FORCE_REDRAW)
import pandas as pd
from IPython.display import display, Image, Markdown
sources = plots.selected_source_table(seed_results)
display(Markdown("### HBC本次绘图实际来源（seed 0）与五seed汇总"))
display(sources[sources.dataset=="HBC"].rename(columns={
    "method":"方法", "protocol":"配置", "epoch":"图中权重轮次", "ARI":"图中单seed ARI",
    "mean":"五seed平均峰值ARI", "std":"五seed标准差", "source_absolute":"实际预测文件"}))
display(Markdown("### 全方法汇总（DLPFC为12切片均值的中位数及切片间标准差）"))
display(pd.read_csv(plots.OUT/"method_summary.csv"))
print("逐seed结果:",plots.OUT/"seed_metrics.csv")
print("当前图件清单:",plots.OUT/"figure_manifest.json")
print("补充对照数据:",plots.SUPPLEMENT)
print("本次显示图件:",len(images),"张；不包含historical_manuscript中的旧图。")


In [ ]:
DISPLAY_SVG = False  # 大型矢量套图浏览器渲染较慢，默认PNG预览；正式图片用同名SVG。
from IPython.display import SVG
for path in images:
    vector = path.with_suffix(".svg")
    if path.name == "SpaSRD_framework_preview.png":
        vector = plots.OUT / "SpaSRD_framework_editable.svg"
    display(Markdown("**"+path.stem+"**\n\n矢量图：`"+str(vector.resolve())+"`"))
    if DISPLAY_SVG:
        display(SVG(filename=str(vector)))
    else:
        display(Image(filename=str(path),width=1000))


## 组合组小区域核查（可选）
比较未加两项正则的基础S与组合S的相同区域裁剪；这里的control是基础S，不是GCN。结果保存于 `results/hbc_combined/refined/figures`，供消融检查，不能替代上面的主方法对照图。


In [ ]:
RUN_REGION_DETAIL = False
if RUN_REGION_DETAIL:
    from src.hbc_combined_figures import generate
    detail_metrics, detail_images = generate(seeds=(0,))
    display(detail_metrics)
    for path in detail_images: display(Image(filename=str(path),width=1000))
